# 8.6 Scalar 优化：不干活的指令也要少占周期

Scalar 单元「不产出数据」，但**FA kernel 的每一拍流水都由它驱动**：循环变量自增、地址偏移计算、`SetFlag/WaitFlag` 同步、分支跳转——全是 Scalar 指令。它们插在 Cube/Vector 指令流里发射，**每一条都在占用发射周期**。

## 1. Scalar 拖慢流水的三种方式

| 方式 | 典型例子 | 后果 |
|--|--|--|
| **量大** | 循环体内每次重算 8 个地址 | 每拍几十条标量指令，发射槽被占满 |
| **依赖长** | 地址算完才能发 Load | Vector/Cube 空等 |
| **分支抖** | 循环里 `if (tail) {...}` | 分支两侧都要发射，流水断流 |

## 2. 手段一：标量上提（Host/Tiling 侧算好）

这是[第3章 Tiling 设计原则](../03_fa_code_architecture/README.md)「Scalar 上提到 Host 计算」在性能维度的再次兑现：

- **凡是不依赖运行时数据的计算，全部上提**：地址表、循环次数、尾块长度、scale 常量；
- kernel 侧从 `TilingData` 读一个现成的数组，比现场算 8 次乘加便宜得多；
- 8.3 的最优 K 切分计算就是典型——整套建模在 Host 完成，kernel 只拿结果。

```cpp
// 反例：kernel 侧现场算
for (int i = 0; i < loopCnt; i++) {
    offset = base + i * stride * col;   // 每轮 2 条标量乘加
    ...
}

// 正例：Tiling 侧预生成偏移表, kernel 侧只自增指针
__gm__ int64_t *offsets = tilingData.offsets;  // Host 算好的表
for (int i = 0; i < loopCnt; i++) {
    DataCopy(dst + offsets[i], ...);           // 1 次读表
}
```

## 3. 手段二：循环内增量化

依赖运行时数据、无法上提的地址计算，改成**每轮只加增量**：

```cpp
// 反例：每轮完整计算
for (i...) addr = base + i * stride;      // 乘法 + 加法

// 正例：循环外算初值, 循环内自增
int64_t addr = base;                       // 循环外 1 次
for (i...) { ...; addr += stride; }        // 每轮 1 次加法
```

## 4. 手段三：分支消除

分支的代价不是跳转本身，而是**两个分支的代码都要发射、流水要重新充满**。消除手段：

| 手段 | 做法 |
|--|--|
| **掩码代替分支** | `if (idx < valid) x = v; else x = 0;` → 乘上 `(idx < valid)` 的 0/1 掩码 |
| **查表代替多路分支** | switch 三分支 → 数组索引一次取出 |
| **模板特化** | 编译期把 `isTail` 做成模板参数，生成两份无分支代码（配合 TilingKey） |

掩码代替分支与 Vector 的 MaskReg 是同一个哲学：**用数据运算换控制流**。

## 动手实验：分支消除的收益模拟


In [ ]:
# 动手：分支消除 vs 条件分支的指令数对比（抽象模拟）
import numpy as np

n = 4096
rng = np.random.default_rng(42)
valid_len = 4000                          # 有效长度, 后 96 个是尾块

# ---- 方案 A: 循环内条件分支 ----
# 每个元素: 1 条比较 + 1 条分支 + 2 条取值(move) = 4 条标量指令
cost_branch = n * 4

# ---- 方案 B: 掩码消除分支 ----
# 每个元素: 1 条掩码生成(比较) + 1 条掩码乘 = 2 条; 且可向量化成 64/条
# 标量视角
cost_mask_scalar = n * 2
# 向量化视角: 一条 Vector 掩码指令处理 64 个元素
cost_mask_vec = (n / 64) * 2

print(f"元素数 {n}, 有效 {valid_len}")
print(f"方案A 条件分支:      {cost_branch:>8.0f} 条指令")
print(f"方案B 掩码(标量):    {cost_mask_scalar:>8.0f} 条指令")
print(f"方案B 掩码(Vector):  {cost_mask_vec:>8.0f} 条指令")
print(f"Vector 化后削减: {1 - cost_mask_vec/cost_branch:.1%}")


### 观察结论

掩码 + 向量化后，控制流开销压缩了一个数量级。注意方案 B 的正确性前提：无效位置的值必须是「乘 0 后无害」的（累加、softmax 的分母都满足）。

## 5. 手段四：同步指令精简

`SetFlag/WaitFlag` 是 FA kernel 里数量最多的标量指令之一（每拍流水至少一对）。精简方向：

- **能合并的等待不拆**：两个输入都就绪的 `WaitFlag` 若硬件支持事件合并，就合并；
- **不写多余的保护**：上一拍已经 `Wait` 过、本拍没有新写的地方，不要重复 `Wait`；
- 与 8.4 的指令编排联动：**同步点越少，软件双缓冲的交错空间越大**。

## 6. 一个 checklist：kernel 侧 Scalar 体检

```
□ 循环体内有没有乘法？（应全部变成增量加法）
□ 循环体内有没有运行时可算的分支？（应掩码化或模板特化）
□ TilingData 里还缺哪些预计算表？
□ SetFlag/WaitFlag 有没有成对冗余？
□ 循环能不能整体展开（次数 ≤ 8 时通常值得）？
```

## 本节要点

- Scalar 指令不产出数据，但占发射周期、拉长依赖链、分支断流水；
- 四个手段：**上提（Tiling 算好）、增量化、分支消除（掩码/查表/模板特化）、同步精简**；
- 掩码代替分支 = 用数据运算换控制流，与 Vector 的 MaskReg 同源；
- 优化顺序：先上提、再增量化、最后抠同步——上提的收益最大且零风险。

## 小测验

1. Scalar 指令「不干活」，为什么会拖慢 Cube 和 Vector？
2. `addr = base + i * stride` 在循环内怎样改写？写出改写前后的指令数对比。
3. 三种分支消除手段各适用什么场景？掩码方案的正确性前提是什么？
4. 为什么说「标量上提」是收益最大且零风险的优化？

>>> 参考答案见 [answer/08.06_answer.txt](answer/08.06_answer.txt)
